<a href="https://colab.research.google.com/github/mundundan-star/online-retail-customer-analytics/blob/main/2.0%20Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The primary purpose of this notebook is to control what features are fed in a particular notebook\model.
To ensure convenient feature_engineering from transaction items, particularly for machine learning training and production, python functions were created and stored in an AWS S3 bucket. This is to esnure that all notebooks beyond this one, only pull data from the database through a consistent channel, using the `v_clean_sales_analytics` via the functions tailored each notebook's needs. That said, this notebooks purpose is the creation of a `feature_engineering` function that will deliver features for revenue, and customer churn, prediction, as well a `customer_segmentation_features` which is depended on data produced by the features engineering, to serve the customer segmentation training and assignments.

In [16]:
!pip install boto3

In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import userdata

from sqlalchemy import create_engine, text, inspect

import boto3
import sys
import os
import importlib

bucket_name = 'sales-data-analytics-portfolio-2026'

#Intializing s3 client with credentials
s3 = boto3.client(
    "s3",
    aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name = 'eu-north-1'
)

with open("feature_engineering.py", "w") as f:
    f.write("""
import pandas as pd
import numpy as np
def feature_engineering(df):

    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    end_date = df["InvoiceDate"].max()

    cust_data = df.groupby("CustomerID")[["InvoiceDate", "StockCode", "Quantity", "UnitPrice", "Revenue"]].agg(
       FirstPurchase = ("InvoiceDate", "min"),
       LastPurchase = ("InvoiceDate", "max"),
       Frequency = ("InvoiceDate", "nunique"),
       ProductDiversity = ("StockCode", "nunique"),
       AvgUnitPrice = ("UnitPrice", "mean"),
       AvgQuantity = ("Quantity", "mean"),
       AOV = ("Revenue", "mean"),
       TotalRevenue = ("Revenue", "sum")
    )

    cust_data["CohortMonth"] = cust_data["FirstPurchase"].dt.to_period("M")
    cust_data["Recency"] = 1 + (end_date - cust_data["LastPurchase"]).dt.days
    cust_data["Tenure"] = 1 + (end_date - cust_data["FirstPurchase"]).dt.days
    cust_data["ObservedLifeSpan"] = 1 + (cust_data["LastPurchase"] - cust_data["FirstPurchase"]).dt.days

    cust_data.drop(["FirstPurchase","LastPurchase"], axis = 1, inplace = True)

    cust_data["RecencyToTenure"] = cust_data["Recency"]/(cust_data["Tenure"])
    cust_data["ActivePurchaseDensity"] = cust_data["Frequency"]/cust_data["ObservedLifeSpan"]
    cust_data["LifetimePurchaseDensity"] = cust_data["Frequency"]/cust_data["Tenure"]

    cust_data["ActiveMRate"] = cust_data["TotalRevenue"]/cust_data["ObservedLifeSpan"]
    cust_data["LifetimeMRate"] = cust_data["TotalRevenue"]/cust_data["Tenure"]

    ipi = (df[["CustomerID", "InvoiceDate"]].sort_values(["CustomerID", "InvoiceDate"], ascending = False))
    ipi = ipi.groupby(["CustomerID", "InvoiceDate"]).size().reset_index()
    ipi["InterPurchaseInterval"] = np.where(ipi["CustomerID"].shift(1) == ipi["CustomerID"], (ipi["InvoiceDate"] - ipi["InvoiceDate"].shift(1)).dt.days + 1, 1)
    ipi = ipi.groupby("CustomerID").agg(
        AvgIPI = ("InterPurchaseInterval", "mean")
    ).reset_index()

    cust_data = cust_data.merge(ipi[["CustomerID", "AvgIPI"]], on = "CustomerID", how = "left")

    cust_data["RelativeSilence"] = cust_data["Recency"]/cust_data["AvgIPI"]

    return cust_data"""
)


# Uploading file to s3 bucket
s3.upload_file('feature_engineering.py', bucket_name, 'functions/feature_engineering.py')
print(f'Successfully uploaded file to {bucket_name} bucket')

s3.download_file(bucket_name, 'functions/feature_engineering.py', 'feature_engineering.py')

sys.path.append(os.getcwd())

import feature_engineering

importlib.reload(feature_engineering)

from feature_engineering import feature_engineering

engine = create_engine(userdata.get("NEON_DATABASE_URL"))

# Feature engineering for train and test sets, product_set
query = """
SELECT *
FROM v_clean_sales_analytics
"""

from datetime import datetime

df = pd.read_sql(text(query), con = engine)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

cutoff_date = '2011-08-31'
cutoff_date = datetime.strptime(cutoff_date, '%Y-%m-%d')


#Customer Segmentation features

df0 = feature_engineering(df[df["InvoiceDate"].astype('str') <= '2011-08-31'])
print('Preview of Features')
print(df0.info())

Successfully uploaded file to sales-data-analytics-portfolio-2026 bucket
Preview of Features
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3317 entries, 0 to 3316
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype    
---  ------                   --------------  -----    
 0   CustomerID               3317 non-null   float64  
 1   Frequency                3317 non-null   int64    
 2   ProductDiversity         3317 non-null   int64    
 3   AvgUnitPrice             3317 non-null   float64  
 4   AvgQuantity              3317 non-null   float64  
 5   AOV                      3317 non-null   float64  
 6   TotalRevenue             3317 non-null   float64  
 7   CohortMonth              3317 non-null   period[M]
 8   Recency                  3317 non-null   int64    
 9   Tenure                   3317 non-null   int64    
 10  ObservedLifeSpan         3317 non-null   int64    
 11  RecencyToTenure          3317 non-null   float64  
 12  ActivePurch

/content/feature_engineering.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


In [18]:
# creating function to pull features specifically for customer Segmentation from dataframe after passing feature_engineering function

import sys
import os

with open("segmentation_features.py", "w") as f:
    f.write("""def segmentation_features(df):
    features = [
    "CustomerID",
    "Tenure",
    "ObservedLifeSpan",
    "TotalRevenue",
    "Recency",
    "ProductDiversity",
    "Frequency"]

    return df[features]""")

s3.upload_file("segmentation_features.py", bucket_name, "functions/segmentation_features.py")
print("Successfully uploaded Segmentation function to S3 Bucket")

s3.download_file(bucket_name, "functions/segmentation_features.py", "segmentation_features.py")
print("\nSuccessfully download Segmentation function from AWA S3")

sys.path.append(os.getcwd())
import segmentation_features

importlib.reload(segmentation_features)

from segmentation_features import segmentation_features

df1 = segmentation_features(df0)
print("\nPreview of features particular to Customer Segmentation")
df1.head()

Successfully uploaded Segmentation function to S3 Bucket

Successfully download Segmentation function from AWA S3

Preview of features particular to Customer Segmentation


,CustomerID,Tenure,ObservedLifeSpan,TotalRevenue,Recency,ProductDiversity,Frequency
0,12346.0,226,1,77183.60,226,1,1
1,12347.0,268,239,2790.86,30,82,5
2,12348.0,259,111,1487.24,149,22,3
3,12350.0,211,1,334.40,211,17,1
4,12352.0,197,35,1561.81,163,26,4


In [19]:
#Feature Engineering specific to Churn Predictions Notebook

with open("churn_label_assignment.py", "w") as f:
    f.write("""import pandas as pd
import numpy as np
def churn_label_assignment(df):
        from datetime import datetime
        df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

        cutoff_date = '2011-08-31'
        cutoff_date = datetime.strptime(cutoff_date, '%Y-%m-%d')

        data1 = df[df["InvoiceDate"] <= cutoff_date]
        data2 = df[df["InvoiceDate"] > cutoff_date]

        # Last purchase before 2011-08-31
        lastpurchase = data1.groupby("CustomerID").agg(
        LastPurchase = ("InvoiceDate","max"))

        # First purchase after 2011-08-31
        firstpurchase = data2.groupby("CustomerID").agg(
        FirstPurchase = ("InvoiceDate","min"))

        data = lastpurchase.join(firstpurchase, how = 'left')

        cutoff_date = data1["InvoiceDate"].max()

        # InterPurchaseInterval to know if customer had churned by August custoff
        data["InterPurchaseInterval1"] = (cutoff_date - data["LastPurchase"]).dt.days

        # InterPurchaseInterval to determine churn after August cutoff
        data["InterPurchaseInterval2"] = (data["FirstPurchase"] - data["LastPurchase"]).dt.days

        churn_days = 100
        data["Inactive"] = np.where(data["InterPurchaseInterval1"] < churn_days, 0, 1)
        data["Churned"] = np.where(data["InterPurchaseInterval2"] < churn_days, 0, 1)

        data = data["Churned"].reset_index()

        print()
        print("Churn labels for transactions before 2011-08-31")
        return data""")

s3.upload_file("churn_label_assignment.py", bucket_name, "functions/churn_label_assignment.py")
print("Successfully uploaded churn function to AWS S3 bucket")

s3.download_file(bucket_name, "functions/churn_label_assignment.py", "churn_label_assignment")
print("Successfully downloaded churn function from AWS S3 bucket for preview")

sys.path.append(os.getcwd())
import churn_label_assignment
importlib.reload(churn_label_assignment)
from churn_label_assignment import churn_label_assignment

churn_label_assignment(df)

Successfully uploaded churn function to AWS S3 bucket
Successfully downloaded churn function from AWS S3 bucket for preview

Churn labels for transactions before 2011-08-31


,CustomerID,Churned
0,12346.0,1
1,12347.0,0
2,12348.0,1
3,12350.0,1
4,12352.0,1
...,...,...
3312,18280.0,1
3313,18281.0,1
3314,18282.0,1
3315,18283.0,0
